# 04 Fine-tune the improved ViT image-only model (GPU, ~30 min per seed)
Uses the best input from notebook 03. Changes from B3 (val AUC 0.530), all in `configs/vo_base.yaml`:
- **v2 cache:** the tumour is not clipped (in v1, roughly half of the I-SPY tumour was pure white).
- **Normalisation** (`NORM`) and **tumour-zoom crop**: the tumour fills ~60% of the input instead of ~20-40%.
- **Central tumour slices** instead of every slice down to edge slivers.
- **Patch-token mean pooling** instead of MAE's untrained [CLS] token.
- **LP-FT:** 2 epochs head-only, then the top 4 blocks unfreeze with their own warm-up.
- **Plain BCE** loss, and several seeds averaged (an ensemble), because one val AUC on 177 patients is noisy.

`RUN_CONTROL = True` also runs `vo_b3_repro`, the old B3 recipe through this code. It should land near 0.53,
which shows that any gain comes from the changes rather than the new code.

1. Session options: **GPU T4 x2 or P100**, Internet **ON**, secret `GITHUB_TOKEN`; attach `mnj-cache2-*` (or `mnj-cache-*`).
2. First `DEBUG = True` (2 epochs, 48 patients), then the real run: **Save & Run All (Commit)**.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # "A", "B" or "C": results go to results/<PERSON>/
CONFIG = "vo_base"          # vo_base | vo_dinov2 | vo_b3_repro
NORM = "kinetic"            # from notebook 03's top rows
CROP = "tumor"
BACKBONE = "mae"            # mae | dinov2_b | vit_sup | dino_b16 | dinov2_s
SEEDS = [0, 1, 2]
EXTRA = []                  # more overrides, e.g. ["model.train_blocks=6", "optim.lr_backbone=2.0e-5"]
RUN_CONTROL = False         # also run vo_b3_repro (seed 0)
DEBUG = False
PUSH_RESULTS = False
NOTEBOOK_NAME = "finetune"

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
assert PERSON in ("A", "B", "C"), 'set PERSON = "A", "B" or "C" in the parameters cell'
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import glob, pandas as pd
def run(config, seed, sets):
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs",
           "--results-dir", f"/kaggle/working/results/{PERSON}", "--set", *sets]
    if DEBUG: cmd.append("--debug")
    print(" ".join(cmd)); subprocess.run(cmd, check=True)
    d = sorted(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime)[-1]
    return d, json.load(open(f"{d}/metrics.json"))

sets = [f"data.norm={NORM}", f"data.crop={CROP}", f"model.backbone={BACKBONE}", *EXTRA]
runs = [run(CONFIG, s, sets) for s in SEEDS]
if RUN_CONTROL:
    runs.append(run("vo_b3_repro", 0, ["model.backbone=mae"]))

In [ ]:
from src import stats
rows = [{"run": os.path.basename(d.rstrip("/")), "val_auc": m["val"]["auc"], "ci": m["val"]["auc_ci"],
         "within_cohort": m["val"]["auc_within"], **{f"{c}": v["auc"] for c, v in m["val"]["by_cohort"].items()},
         "best_epoch": m["best_epoch"], "gpu_h": m["gpu_hours"]} for d, m in runs]
display(pd.DataFrame(rows).round(3))
main_runs = [d for d, m in runs[:len(SEEDS)]]
p = pd.concat([pd.read_csv(f"{d}/val_preds.csv").set_index("pid") for d in main_runs], axis=1, keys=range(len(main_runs)))
ens = pd.DataFrame({"label": p[0].label, "cohort": p[0].cohort, "prob": p.xs("prob", axis=1, level=1).mean(1)})
s = stats.summary(ens.label, ens.prob)
by = {c: stats.auc(g.label, g.prob) for c, g in ens.groupby("cohort")}
print(f"ENSEMBLE of {len(main_runs)} seeds: val AUC {s['auc']:.3f} (95% CI {s['auc_ci'][0]:.3f}-{s['auc_ci'][1]:.3f}), "
      f"AUPRC {s['auprc']:.3f} | " + " ".join(f"{c} {v:.3f}" for c, v in by.items()))
print("references (val): old B3 ViT 0.530 | clinical LR B0 0.639 | clinical + size B1 0.653-0.660 | F2 fusion 0.690")
tag = f"vo_{BACKBONE}_{NORM}_{CROP}_ensemble"
os.makedirs(f"/kaggle/working/results/{PERSON}/{tag}", exist_ok=True)
ens.to_csv(f"/kaggle/working/results/{PERSON}/{tag}/val_preds.csv")
json.dump({"runs": [os.path.basename(d.rstrip("/")) for d in main_runs], "val": s, "by_cohort": by},
          open(f"/kaggle/working/results/{PERSON}/{tag}/metrics.json", "w"), indent=2, default=float)

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not DEBUG:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")